In [1]:
#!/usr/bin/env python3
"""
FoodEVPred v2 -- label assignment and train/test split

Input   FoodEVPred_cdhit40_master.csv   (8,509 CD-HIT-40 cluster representatives)
Output  splits/train.csv                (80%)
        splits/test.csv                 (20%, held out for final evaluation only)
        splits/train_ids.txt , test_ids.txt
        splits/split_summary.csv

LABELS
------
primary_label   the three-class task, same encoding as the original submission
                  0 = Non_EV     1 = Milk_EV     2 = Plant_EV

secondary_label a two-digit code:  TENS = species, UNITS = polarity
                  11 Arabidopsis EV      10 Arabidopsis non-EV
                  21 Brassica EV         20 Brassica non-EV
                  31 human milk EV       30 human non-EV
                  41 bovine milk EV      40 bovine non-EV
                  51 donkey milk EV      50 donkey non-EV

                One column, three facts, no lookup table needed:
                  secondary_label // 10  -> species id   (leave-one-species-out)
                  secondary_label %  10  -> 1 = EV, 0 = non-EV
                  the pair (species, polarity) -> the within-species binary task

SPLIT
-----
Stratified 80:20 on secondary_label, so every one of the ten class-by-polarity
groups is split 80:20 independently. Stratifying on primary_label alone would
let donkey (193 positives) drift, since it is only 6% of Milk_EV.

No group constraint is needed: every sequence is a CD-HIT-40 cluster
representative, so no two sequences in the dataset exceed 40% identity and no
train/test pair can either.

The test set is touched ONCE, at the end, on the final pipeline. All feature
selection, hyperparameter tuning and model selection happen inside the training
set, which is further split 80:20 into fit/validation at each of those steps.
"""
import os
import pandas as pd
from sklearn.model_selection import train_test_split

SEED = 42
TEST_FRACTION = 0.20
MASTER = "/kaggle/input/datasets/harshiikkaa/foodev-v3/foodev_final_master_file_13760.csv"
OUT = "splits"
os.makedirs(OUT, exist_ok=True)

PRIMARY = {"Non_EV": 0, "Milk_EV": 1, "Plant_EV": 2}

# tens digit = species, units digit = polarity (1 = EV, 0 = non-EV)
SECONDARY = {
    "arabidopsis_apoplastic_EV":           11,
    "arabidopsis_apoplastic_EV__NEGATIVE": 10,
    #"brassica_apoplastic_EV":              21,
    #"brassica_apoplastic_EV__NEGATIVE":    20,
    "human_milk_EV":                       31,
    "human_milk_EV__NEGATIVE":             30,
    "bovine_milk_EV":                      41,
    "bovine_milk_EV__NEGATIVE":            40,
    "donkey_milk_EV":                      51,
    "donkey_milk_EV__NEGATIVE":            50,
}
SPECIES = {1: "Arabidopsis thaliana", 2: "Brassica oleracea",
           3: "Homo sapiens", 4: "Bos taurus", 5: "Equus asinus"}

df = pd.read_csv(MASTER, low_memory=False)
df["primary_label"] = df["class_label"].map(PRIMARY)
df["secondary_label"] = df["subclass"].map(SECONDARY)
assert df[["primary_label", "secondary_label"]].notna().all().all(), "unmapped label"
df["secondary_label"] = df["secondary_label"].astype(int)

keep = ["seq_id", "sequence", "primary_label", "secondary_label"]

train, test = train_test_split(
    df, test_size=TEST_FRACTION, random_state=SEED,
    stratify=df["secondary_label"])       # stratify on the finest grouping

train[keep].sort_values("seq_id").to_csv(f"{OUT}/train.csv", index=False)
test[keep].sort_values("seq_id").to_csv(f"{OUT}/test.csv", index=False)
train[["seq_id"]].to_csv(f"{OUT}/train_ids.txt", index=False, header=False)
test[["seq_id"]].to_csv(f"{OUT}/test_ids.txt", index=False, header=False)

rows = []
for sec, g in df.groupby("secondary_label"):
    rows.append(dict(secondary_label=sec,
                     species=SPECIES[sec // 10],
                     polarity="EV" if sec % 10 else "non-EV",
                     subclass=g["subclass"].iloc[0],
                     primary_label=int(g["primary_label"].iloc[0]),
                     total=len(g),
                     train=int((train["secondary_label"] == sec).sum()),
                     test=int((test["secondary_label"] == sec).sum())))
summary = pd.DataFrame(rows).sort_values("secondary_label")
summary.to_csv(f"{OUT}/split_summary.csv", index=False)

print(summary.to_string(index=False))
print(f"\nTRAIN {len(train):,}   TEST {len(test):,}   seed {SEED}")
print("\nprimary_label distribution")
print(pd.crosstab(df["primary_label"],
                  df["seq_id"].isin(test["seq_id"]).map({False: "train", True: "test"})
                  ).to_string())
print("\nTest set is evaluated ONCE, on the final pipeline. Every fitted step "
      "(scaler, feature selection, hyperparameters, model choice) is fitted "
      "inside the training set only.")


 secondary_label              species polarity                            subclass  primary_label  total  train  test
              10 Arabidopsis thaliana   non-EV arabidopsis_apoplastic_EV__NEGATIVE              0   5411   4329  1082
              11 Arabidopsis thaliana       EV           arabidopsis_apoplastic_EV              2   2131   1705   426
              30         Homo sapiens   non-EV             human_milk_EV__NEGATIVE              0   2022   1617   405
              31         Homo sapiens       EV                       human_milk_EV              1    845    676   169
              40           Bos taurus   non-EV            bovine_milk_EV__NEGATIVE              0   1909   1527   382
              41           Bos taurus       EV                      bovine_milk_EV              1    746    597   149
              50         Equus asinus   non-EV            donkey_milk_EV__NEGATIVE              0    535    428   107
              51         Equus asinus       EV          